# 02b. Aplicar Splink

Carrega o JSON do [`02_treinar_splink.ipynb`](02_treinar_splink.ipynb) e pontua
pares (`predict`, `p ≥ 0,5`) no conjunto inteiro. Sem clustering. As 12 regras
de blocking de predição estão neste notebook (não no treino).

Saída: `splink_predictions.parquet` (ids + score). Avaliação no
[`03_avaliar.ipynb`](03_avaliar.ipynb); lista única no
[`04_atribuir.ipynb`](04_atribuir.ipynb).


In [ ]:
import json
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from IPython.display import display
from config import (
    DUCKDB_MEMORY_LIMIT,
    DUCKDB_THREADS,
    PREDICT_NUM_CHUNKS_LEFT,
    PREDICT_NUM_CHUNKS_RIGHT,
    SPLINK_INPUT_VIEW,
    SPLINK_MODEL_JSON,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA,
    TABELA_CPF_LIMPA,
    drop_splink_temp_tables,
    ensure_output_dir,
    get_connection,
    get_splink_db_api,
    materialize_splink_input,
    print_paths,
    require_tables,
)

print_paths()
if not SPLINK_MODEL_JSON.exists():
    raise RuntimeError(
        f'Modelo Splink não encontrado: {SPLINK_MODEL_JSON}. '
        'Rode notebooks/02_treinar_splink.ipynb — este notebook não retreina.'
    )

con = get_connection()
drop_splink_temp_tables(con)
require_tables(con, [TABELA_CENSO_LIMPA, TABELA_CPF_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
db_api = get_splink_db_api(con)

n_reg = con.execute(f'SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW}').fetchone()[0]
duck_settings = con.execute(
    "SELECT current_setting('threads'), current_setting('memory_limit')"
).fetchone()
print(f'Registros: {n_reg:,}')
print(
    f'DuckDB: threads={duck_settings[0]}, memory_limit={duck_settings[1]} '
    f'(defaults: {DUCKDB_THREADS}, {DUCKDB_MEMORY_LIMIT})'
)
print('Modelo:', SPLINK_MODEL_JSON)
print('Predict chunks L/R:', PREDICT_NUM_CHUNKS_LEFT, PREDICT_NUM_CHUNKS_RIGHT)


## Blocking de predição

Doze regras **OR** para gerar candidatos no `predict` (recall). Não são o
blocking do EM nem o do prior (esses ficam no 02).

`nome_meio` e `cpf_norm` ficam de fora. Quem não tem meio sumiria do predict;
CPF na predição faria a GT sempre candidata e o 03 circularia.
CEP em quatro regras (não só DOB+CEP). Sexo em `ultimo+mes+dia+cep` e
`DOB+UF+sexo+cep`, não nas regras com primeiro+último.


In [ ]:
from splink import block_on

blocking_rules = [
    block_on('nome_completo_phon'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'ano_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'mes_nascimento', 'dia_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'mes_nascimento', 'ano_nascimento'),
    block_on('primeiro_nome_phon', 'data_nascimento'),
    block_on('ultimo_nome_phon', 'data_nascimento'),
    block_on('primeiro_nome_phon', 'mes_nascimento', 'dia_nascimento','cep'),
    block_on('primeiro_nome_phon', 'mes_nascimento', 'ano_nascimento','cep'),
    block_on('ultimo_nome_phon', 'mes_nascimento', 'dia_nascimento', 'sexo', 'cep'),
    block_on('ultimo_nome_phon', 'mes_nascimento', 'ano_nascimento','cep'),
    block_on('data_nascimento', 'cep'),
    block_on('data_nascimento', 'uf', 'sexo','cep'),
]


In [ ]:
from splink.blocking_analysis import (
    cumulative_comparisons_to_be_scored_from_blocking_rules_data,
)
from splink.internals.charts import cumulative_blocking_rule_comparisons_generated

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')
n_censo_chart = con.execute('SELECT COUNT(*) FROM splink_censo').fetchone()[0]
n_cpf_chart = con.execute('SELECT COUNT(*) FROM splink_cpf').fetchone()[0]
print(f'Chart blocking (conjunto inteiro): censo {n_censo_chart:,} | cpf {n_cpf_chart:,}')

df_blocking = cumulative_comparisons_to_be_scored_from_blocking_rules_data(
    table_or_tables=['splink_censo', 'splink_cpf'],
    blocking_rules=blocking_rules,
    db_api=db_api,
    link_type='link_only',
)
n_pares_blocking = int(df_blocking['cumulative_rows'].iloc[-1])
print(f'Pares OR: {n_pares_blocking:,}')
cumulative_blocking_rule_comparisons_generated(df_blocking.to_dict(orient='records'))


Linker no JSON do 02 + as 12 regras acima. Views = `censo_limpo` / `cpf_limpo` inteiros.


In [ ]:
from splink import Linker

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')
n_censo_limpo = con.execute(f'SELECT COUNT(*) FROM {TABELA_CENSO_LIMPA}').fetchone()[0]
n_cpf_limpo = con.execute(f'SELECT COUNT(*) FROM {TABELA_CPF_LIMPA}').fetchone()[0]
n_censo_view = con.execute('SELECT COUNT(*) FROM splink_censo').fetchone()[0]
n_cpf_view = con.execute('SELECT COUNT(*) FROM splink_cpf').fetchone()[0]
assert n_censo_view == n_censo_limpo and n_cpf_view == n_cpf_limpo, (
    f'Linker não está no conjunto inteiro: '
    f'censo {n_censo_view} vs limpo {n_censo_limpo}; '
    f'cpf {n_cpf_view} vs limpo {n_cpf_limpo}'
)
print('Linker: conjunto inteiro', f'{n_censo_view:,}', f'{n_cpf_view:,}')

with open(SPLINK_MODEL_JSON, 'r') as file:
    data = json.load(file)
data['retain_intermediate_calculation_columns'] = False
data['retain_matching_columns'] = False
data['blocking_rules_to_generate_predictions'] = blocking_rules

linker = Linker(
    ['splink_censo', 'splink_cpf'],
    data,
    db_api=db_api,
    input_table_aliases=['censo', 'cpf'],
)


`predict(0.5)` gera candidatos. Avaliação no 03.


In [ ]:
predict_kwargs = {'threshold_match_probability': 0.5}
if PREDICT_NUM_CHUNKS_LEFT is not None:
    predict_kwargs['num_chunks_left'] = PREDICT_NUM_CHUNKS_LEFT
if PREDICT_NUM_CHUNKS_RIGHT is not None:
    predict_kwargs['num_chunks_right'] = PREDICT_NUM_CHUNKS_RIGHT

df_predict = linker.inference.predict(**predict_kwargs)
pred = df_predict.physical_name
print('predictions table:', pred)
print('pares:', con.execute(f'SELECT COUNT(*) FROM {pred}').fetchone()[0])


Parquet estreito (ids + score). O largo do Splink só vive até o COPY.


In [ ]:
ensure_output_dir()
cols = [r[0] for r in con.execute(f'DESCRIBE {pred}').fetchall()]
keep = [
    c for c in ('unique_id_l', 'unique_id_r', 'match_probability', 'match_weight')
    if c in cols
]
con.execute(
    f"COPY (SELECT {', '.join(keep)} FROM {pred}) "
    f"TO '{SPLINK_PREDICTIONS}' (FORMAT PARQUET, COMPRESSION ZSTD)"
)
drop_splink_temp_tables(con)
con.execute('CHECKPOINT')
print('Predictions:', SPLINK_PREDICTIONS)
print('colunas:', keep)


Distribuição de score no parquet (sem rótulos). Discordar nome/DOB deve penalizar bits, não ≈ 0.


In [ ]:
pred_parquet = f"read_parquet('{SPLINK_PREDICTIONS}')"
agg = [
    'COUNT(*) AS n',
    'MIN(match_probability) AS min_p',
    'AVG(match_probability) AS avg_p',
    'MAX(match_probability) AS max_p',
]
if 'match_weight' in keep:
    agg.extend(
        [
            'MIN(match_weight) AS min_w',
            'AVG(match_weight) AS avg_w',
            'MAX(match_weight) AS max_w',
        ]
    )
display(con.execute(f'SELECT {", ".join(agg)} FROM {pred_parquet}').df())

if 'match_weight' in keep:
    # DuckDB 1.x não tem width_bucket. 20 faixas de 3 bits em [-20, 40].
    display(
        con.execute(f'''
        SELECT
            faixa,
            CASE WHEN faixa <= 0 THEN NULL ELSE -20 + 3 * (faixa - 1) END AS peso_min,
            CASE WHEN faixa >= 21 THEN NULL ELSE -20 + 3 * faixa END AS peso_max,
            n_pares
        FROM (
            SELECT
                CASE
                    WHEN match_weight < -20 THEN 0
                    WHEN match_weight > 40 THEN 21
                    WHEN match_weight = 40 THEN 20
                    ELSE CAST(FLOOR((match_weight + 20) / 3.0) AS INTEGER) + 1
                END AS faixa,
                COUNT(*) AS n_pares
            FROM {pred_parquet}
            WHERE match_weight IS NOT NULL
            GROUP BY 1
        )
        ORDER BY 1
        ''').df()
    )


In [ ]:
for label, p in [
    ('modelo', SPLINK_MODEL_JSON),
    ('predictions', SPLINK_PREDICTIONS),
]:
    print(f'{label:12s} {"ok " if p.exists() else "FALTA"} {p}')
con.close()
